# Suite BLW — A cache layer: a store given a layer below

A store may be given the layer below it, `Builders.Store(below)`, which makes it a cache layer ([CAP-283: a store given a store below](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-283); [the stacked stores proposal, a cache layer and the layer below](https://github.com/pitaman71/mbse-basis/blob/main/proposals/STACKED-STORES.md#a-cache-layer-and-the-layer-below), its second stage). Its transaction opens one transaction of the layer below for its whole duration. Each input frame its `exec` takes, it interprets read-only over what it holds, as far as it can know that what it holds is all there is: where it cannot, the frame halts, and goes to the layer below's transaction with the rest of the expression, writing; a frame it completes goes down too. Each frame that comes back is given: one annotated with an error as it is; one complete, its journal applied to what the cache layer holds; one partial, executed again by the cache layer, writing. The cache layer loads nothing on a read: it holds what the journals of the frames that come back, its own writes, and the layer below's follow give it. At its commit, the layer below's transaction commits first, a refusal there refusing both. An install runs below first, then in the cache layer. What another commits below reaches it by the follow, applied to the objects it holds alone ([CAP-285: writes through, others' writes in](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-285)). Given no layer below, a store holds what is written to it, as it always has: the loopback.

Python and TypeScript run this suite with the same case IDs. The case setup is below.

In [ ]:
import { Builders as B, Changes, JSON as Json, Plain, Provenance, Transactions } from "@mbse/schemas/Framework";
import { LookupError, ValueError } from "@mbse/schemas/Framework/Errors";
import { assert, equal, map, raises } from "./support.js";

const { Frame } = Transactions;

/** crm.Person, owned by the singleton crm.TheBook under a key, with a name, an age and a home, who knows persons and
 * keeps pets, each owned by its keeper under a name; crm.Note, counted. */
function crm(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("crm/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Knows: S.OfRelation, Person: S.OfObject, Holds: S.OfRelation, Book: S.OfObject, Home: S.OfObject,
      Note: S.OfObject, Pet: S.OfObject, Keeps: S.OfRelation })
    .spec([
      (_: any) => _.Knows.name("crm.Knows").links([(t: any) => t.name("knower"), (t: any) => t.name("known")]),
      (_: any) => _.Home.name("crm.Home").properties({ city: S.OfStr }),
      (_: any) => _.Note.name("crm.Note").ref(true).properties({ text: S.OfStr }),
      (_: any) => _.Keeps.name("crm.Keeps").links([(t: any) => t.name("keeper"), (t: any) => t.name("pet")]).properties({
        name: S.OfStr }).uniques([["keeper", "name"], ["pet"]]),
      (_: any) => _.Pet.name("crm.Pet").ref(true).properties({ kind: S.OfStr }).adjacencies([
        (t: any) => t.name("keeper").relation([_.Keeps]).me("pet").owner(true)]),
      (_: any) => _.Person.name("crm.Person").ref(true).properties({ name: S.OfStr, age: S.OfInt, home: _.Home })
        .adjacencies([
          (t: any) => t.name("knows").relation([_.Knows]).me("knower"),
          (t: any) => t.name("known_by").relation([_.Knows]).me("known"),
          (t: any) => t.name("book").relation([_.Holds]).me("contact").owner(true),
          (t: any) => t.name("pets").relation([_.Keeps]).me("keeper"),
        ]),
      (_: any) => _.Holds.name("crm.Holds").links([(t: any) => t.name("book"), (t: any) => t.name("contact")]).properties({
        key: S.OfStr }).uniques([["book", "key"], ["contact"]]),
      (_: any) => _.Book.name("crm.Book").ref(true).singleton("crm.TheBook").adjacencies({
        contacts: (t: any) => t.relation([_.Holds]).me("book") }),
    ])]))]);
}

/** A transaction making the contact `key`, named `name`, of `age`, where given. */
function person(key: string, name: string, age: bigint | null = null): (t: any) => any {
  return (t: any) => t.uid(`person/${key}`).mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ p: t.crm.Person, book: t.crm.Book }).spec([
      (_: any) => _.book.contacts({ [key]: (e: any) => e.contact(_.p) }),
      (_: any) => age === null ? _.p.name(name) : _.p.name(name).age(age)])]))]);
}

/** `store`, holding crm's schemas, and Ada, 36, who knows Bob, and Bob. */
function seeded(store: any): any {
  store.transaction(crm);
  store.transaction(person("bob", "Bob"));
  store.transaction((t: any) => t.uid("ada/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ a: t.crm.Person, b: t.crm.Person, book: t.crm.Book }).spec([
      (_: any) => _.book.contacts({ ada: (e: any) => e.contact(_.a), bob: (e: any) => e.contact(_.b) }),
      (_: any) => _.a.name("Ada").age(36n).knows([_.b])])]))]));
  return store;
}

/** A layer below seeded, and a cache layer over it, made after it, holding its schemas. */
function stacked(): [any, any] {
  const lower = seeded(new B.Store());
  const cache: any = new B.Store(lower);
  cache.transaction(crm);
  return [lower, cache];
}

/** Each frame's bindings, by symbol, as their paths' JSON, the store's own and another's alike, and its errors. */
function paths(store: any, frames: any[]): string {
  return JSON.stringify(frames.map((frame) => [[...frame].sort(([a], [b]) => a < b ? -1 : 1).map(([symbol, target]) =>
    [symbol, Json.dumps((Plain.ToPlain(store)(target.as_schema(), target) as any).get("path"))]),
  (frame.errors ?? []).map((error: any) => error.kind)]));
}

/** Each person the store holds: name, age and whom they know, read within a transaction of its own. */
function people(store: any): unknown[] {
  return store.transaction(() => [...store.extent("crm.Person")].map((p: any) => [p.name.get(), store.member(p, "age"),
    [...p.knows].map((e: any) => e.known.name.get()).sort()]).sort((a: any, b: any) => a[0] < b[0] ? -1 : 1));
}

/** A map of every person. */
function everyone(t: any): any {
  return t.uid("everyone/1").mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols({ x: t.crm.Person })]))]);
}

/** A map of the person named `name`, then an update writing their age. */
function aged(name: string, age: bigint): (t: any) => any {
  return (t: any) => t.uid(`aged/${name}`).mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols(
    { x: t.crm.Person }).spec([(_: any) => _.x.name(name)])]).then([(w: any) => w.update((u: any) => u.apply([
    (p: any) => p.symbols({ x: t.crm.Person }).spec([(_: any) => _.x.age(age)])]))]))]);
}

/** A stream's items, collected as they are given; its error thrown. */
function items(stream: any): any[] {
  const given: any[] = [];
  let failure: { error: unknown } | null = null;
  stream.subscribe({ next: (item: unknown) => given.push(item), error: (error: unknown) => { failure = { error }; } });
  if (failure !== null) throw (failure as { error: unknown }).error;
  return given;
}

/** The frames of a stream's items, its completion markers left out. */
function frames(given: any[]): any[] {
  return given.filter((item) => item instanceof Frame);
}

/** The reference of the contact `key`, a value of `store`'s, by its path. */
function contact(store: any, key: string): any {
  return Plain.FromPlain(store)(store.registered("References.OfObject"), map({ path: {
    singleton: "crm.TheBook", next: { entry: { i: "contacts", o: "book", properties: { key },
      next: { target: {} } } } } }));
}

/** A transaction: the person named `knower` knows the one named `known`. */
function knows(knower: string, known: string): (t: any) => any {
  return (t: any) => t.uid(`knows/${knower}/${known}`).mutations([(mu: any) => mu.map((m: any) => m.symbols([
    (p: any) => p.symbols({ x: t.crm.Person, y: t.crm.Person }).spec([
      (_: any) => _.x.name(knower), (_: any) => _.y.name(known)])]).then([(w: any) => w.update((u: any) => u.apply([
    (p: any) => p.symbols({ x: t.crm.Person, y: t.crm.Person }).spec([(_: any) => _.x.knows([_.y])])]))]))]);
}

/** A journal, a value of `store`'s, as plain data read back as JSON: its transactions. */
function journaled(store: any, journal: unknown): any[] {
  return JSON.parse(Json.dumps(Plain.ToPlain(store)(store.registered("Changes.Journal"), journal) as any));
}

<a id="BLW-01"></a>
## BLW-01 · A store given a layer below is a cache layer, `store._below`: an install runs below first, then in it, one the layer below refuses refused before it runs in it; it loads nothing on a read, holding none of what the layer below held before it, which a visitor of it does not find; a transaction's frames come from below, the same as a store with no layer below gives, each by its reference

In [ ]:
{
  const [lower, cache] = stacked();
  const twin = seeded(new B.Store());
  assert(cache._below === lower && twin._below === null);
  assert(cache.registered("crm.Person").name === "crm.Person" && lower.registered("crm.Person").name === "crm.Person");
  cache.transaction(() => {
    const book = cache.singleton("crm.TheBook");
    assert([...book.contacts].length === 0 && [...cache.extent("crm.Person")].length === 0); // nothing loaded
  });
  const given = cache.transaction(everyone);
  assert(paths(cache, given) === paths(twin, twin.transaction(everyone)) && given.length === 2);
  assert(equal(people(cache), [])); // a frame given back makes nothing present: its object is the layer below's
  const broken = (t: any): any => t.uid("broken/1").mutations([(mu: any) => mu.update((u: any) => u.apply([(p: any) => p.symbols(
    { s: t.Schemas.OfObject }).spec([(_: any) => _.s.name("crm.Other")])]))]);
  raises(ValueError, () => cache.transaction(broken), // an install the layer below refuses: refused, before it runs here
    "symbol 's' of an update is bound by no term before it");
}

<a id="BLW-02"></a>
## BLW-02 · Writes go through: a frame that halts where a term would write goes below with its remainder, and the layer below's transaction writes it; what it made comes back as the frame's journal, applied to what the cache layer holds, within the same transaction, so that an object made through the cache layer is held by it, all of it; the frame given carries the cache layer's journal, under the transaction's uid; what changed only what the cache layer does not hold changes nothing in it

In [ ]:
{
  const [lower, cache] = stacked();
  const twin = seeded(new B.Store());
  const given = cache.transaction((txn: any = null) => {
    const streamed = items(txn.exec(new Frame(), person("cy", "Cy", 30n), true));
    assert(equal([...cache.extent("crm.Person")].map((p: any) => [p.name.get(), cache.member(p, "age")]), [["Cy", 30n]]));
    return streamed; // held within it
  });
  const [made] = frames(given);
  assert(equal(given[given.length - 1], map({ end: "0" })) && made.remainder === null && made.errors.length === 0);
  const [journal] = journaled(cache, made.journal);
  assert(journal.id === "person/cy#0" && equal(journal.changes.map((change: any) => Object.keys(change)),
    [["entry"], ["property"], ["property"]]));
  twin.transaction(person("cy", "Cy", 30n));
  assert(equal(people(cache), [["Cy", 30n, []]]) && equal(people(lower), people(twin))); // held above; written below
  cache.transaction(() => {
    assert([...cache.extent("crm.Person")][0] === cache.singleton("crm.TheBook").contacts({ key: "cy" }));
  });
  assert(cache._knows(cache._present(map({ singleton: "crm.TheBook", next: { entry: {
    i: "contacts", o: "book", properties: { key: "cy" }, next: { target: {} } } } })))); // all of it
  const older = cache.transaction(aged("Ada", 40n)); // Ada, not held above: written below, nothing changed above
  assert(equal(older.map((frame: any) => frame.journal), [null]) && equal(people(cache), [["Cy", 30n, []]]));
  twin.transaction(aged("Ada", 40n));
  assert(equal(people(lower), people(twin)) && paths(cache, older) === paths(twin, twin.transaction((t: any) => t.mutations([
    (mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols({ x: t.crm.Person }).spec([(_: any) => _.x.name("Ada")])]))]))));
  const homed = (t: any): any => t.uid("homed/1").mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols(
    { x: contact(cache, "cy") })]).then([(w: any) => w.update((u: any) => u.apply([(p: any) => p.symbols(
    { x: contact(cache, "cy") }).spec([(_: any) => _.x.home((h: any) => h.city("Oslo"))])]))]))]);
  const [home] = cache.transaction(homed); // a value object it assigns, of the layer below's, given as the cache layer's
  assert(Json.dumps(Plain.ToPlain(cache)(cache.registered("crm.Home"), home.assigns.get("x").get("home")) as any)
    === '{"city": "Oslo"}');
  const nobody = (t: any): any => t.uid("nobody/1").mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols(
    { x: contact(cache, "cy") })]).then([(w: any) => w.update((u: any) => u.apply([(p: any) => p.symbols(
    { x: contact(cache, "cy") }).spec([(_: any) => _.x.age(31n)])]).then([(w2: any) => w2.map((m: any) => m.symbols([
    (p: any) => p.symbols({ x: t.crm.Person, y: t.crm.Person }).spec([(_: any) => _.y.name("Zed")])]))]))]))]);
  const streamed = cache.transaction((txn: any = null) => { // no frame given, below: what it changed comes back on its
    const all = items(txn.exec(new Frame(), nobody, true)); // completion marker
    assert(frames(all).length === 0 && equal([...cache.extent("crm.Person")].map((p: any) =>
      [p.name.get(), cache.member(p, "age")]), [["Cy", 31n]])); // applied within it
    return all;
  });
  const [marked] = journaled(cache, streamed[streamed.length - 1].get("journal"));
  assert(streamed[streamed.length - 1].get("end") === "0" && marked.changes[0].property.set.name === "age");
}

<a id="BLW-03"></a>
## BLW-03 · Errors are forwarded: a frame the layer below annotates with an error is given as it is, its errors as the layer below's interpreter wrote them; the scope then raises the first, and neither layer commits anything

In [ ]:
{
  const [lower, cache] = stacked();
  const twin = seeded(new B.Store());
  const Person = lower.registered("crm.Person");
  const knowsBob = lower.transaction(() => Plain.ToPlain(lower).Transaction(lower.Mutations.OfTransaction((t: any) => t
    .uid("knows/1").mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols({ x: Person }).spec([
      (_: any) => _.x.name("Ada")])]).then([(w: any) => w.update((u: any) => u.apply([(p: any) => p.symbols(
      { x: Person }).spec([(_: any) => _.x.age(1n).knows([lower.singleton("crm.TheBook").contacts(
      { key: "bob" })])])]))]))])))); // a spec naming Bob by his path, built while he is held
  for (const store of [lower, twin]) {
    store.transaction(() => store.crm.Book(store.singleton("crm.TheBook"), (b: any) => b.contacts((a: any) =>
      a.remove((e: any) => e.key("bob")))));
  }
  const seen: any[] = [];
  raises(LookupError, () => cache.transaction((txn: any = null) => {
    seen.push(...items(txn.exec(new Frame(), Plain.FromPlain(cache).Transaction(knowsBob), true)));
  }), "no object of the store has the path a spec refers to");
  const [failed] = frames(seen);
  assert(equal(failed.errors.map((e: any) => [e.subterm, e.kind, e.message]), [
    ["$.mutations[0].map.then[0].update", "LookupError", "no object of the store has the path a spec refers to"]]));
  assert(paths(cache, [failed]) === JSON.stringify([[[["x", Json.dumps(map({ singleton: "crm.TheBook", next: { entry: {
    i: "contacts", o: "book", properties: { key: "ada" }, next: { target: {} } } } }))]], ["LookupError"]]]));
  assert(equal(people(lower), [["Ada", 36n, []]]) && equal(people(cache), [])); // nothing committed in either
  let again: any = null;
  raises(LookupError, () => cache.transaction((txn: any = null) => {
    [again] = frames(items(txn.exec(failed, Plain.FromPlain(cache).Transaction(knowsBob), true))); // annotated: as it is
  }), "no object of the store has the path a spec refers to");
  assert(again.errors.length === 1 && again.errors[0].equals(failed.errors[0]) && again.remainder === null);
}

<a id="BLW-04"></a>
## BLW-04 · The layer below commits first: the cache layer's transaction holds one transaction of the layer below open for its whole duration, which commits within the cache layer's commit, before it; a refusal below, a unique clause broken against an entry the cache layer does not hold, refuses both, and where the cache layer's raises, the layer below's is dropped

In [ ]:
{
  const [lower, cache] = stacked();
  const order: [string, boolean][] = [];
  Changes.observe(lower, () => order.push(["below", cache._proposals !== null]));
  Changes.observe(cache, () => order.push(["cache", cache._proposals !== null]));
  cache.transaction((txn: any = null) => {
    items(txn.exec(new Frame(), person("cy", "Cy"), true));
    assert(lower._proposals !== null); // the layer below's transaction, open as long as the cache layer's
  });
  assert(equal(order, [["below", true], ["cache", false]])); // the layer below committed within the cache layer's commit
  raises(ValueError, () => cache.transaction(person("ada", "Eve")), "breaks unique"); // a second "ada", not held above
  assert(equal(people(lower), [["Ada", 36n, ["Bob"]], ["Bob", null, []], ["Cy", null, []]])
    && equal(people(cache), [["Cy", null, []]]));
  raises(ValueError, () => cache.transaction((txn: any = null) => {
    items(txn.exec(new Frame(), person("dee", "Dee"), true));
    throw new ValueError("stop"); // the cache layer's throws: the layer below's is dropped
  }), "stop");
  assert(lower._proposals === null && equal(people(lower).map((each: any) => each[0]), ["Ada", "Bob", "Cy"]));
  assert(equal(people(cache), [["Cy", null, []]]));
}

<a id="BLW-05"></a>
## BLW-05 · Given no layer below, a store holds what is written to it: its `exec`, read-only, halts a frame that would write, which the same transaction executes again, writing, the loopback; nothing is forwarded, and its frames carry its own journals

In [ ]:
{
  const alone = seeded(new B.Store());
  assert(alone._below === null && alone._lower === null);
  const [halted, looped] = alone.transaction((txn: any = null) => {
    assert(alone._lower === null);
    const [stopped] = frames(items(txn.exec(new Frame(), aged("Ada", 37n), false)));
    const [resumed] = frames(items(txn.exec(stopped, stopped.remainder, true))); // the loopback
    return [stopped, resumed];
  });
  assert(halted.remainder.subterm === "$.mutations[0].map.then[0].update" && looped.remainder === null);
  const [journal] = journaled(alone, looped.journal);
  assert(JSON.stringify(journal.changes) === JSON.stringify([
    { property: { set: { holder: "o0", name: "age", value: 37 } } }]));
  assert(equal(people(alone)[0], ["Ada", 37n, ["Bob"]]));
}

<a id="BLW-06"></a>
## BLW-06 · A partial frame that comes back from a real layer below, as well as from the loopback, is executed again by the cache layer, writing, and its writes are sent below as it makes them: a layer below that gives frames back unfinished, as one interpreting read-only does, still has them written

In [ ]:
{
  /** A layer below whose transaction's `exec` interprets the first frame it is given read-only, whatever it is asked, as
   * a layer that gives frames back unfinished does, once `armed`: what it gives back partial, kept in `partial`. */
  class Halting extends B.Store {
    armed = false;
    readonly partial: any[] = [];

    override _scope<T>(body: (txn: any) => T): T {
      return super._scope((txn: any) => {
        const given = txn.exec.bind(txn);
        let asked = 0;
        txn.exec = (frame: unknown, remainder: unknown, allowWrite: boolean): any => {
          asked += 1;
          const stream = given(frame, remainder, allowWrite && (asked > 1 || !this.armed));
          return new Transactions.Observable<any>((observer) => {
            stream.subscribe({
              next: (item: any) => {
                if (item instanceof Frame && item.remainder !== null) this.partial.push(item);
                observer.next!(item);
              },
              error: (error: unknown) => observer.error!(error),
              complete: () => observer.complete!(),
            });
          });
        };
        return body(txn);
      });
    }
  }
  const lower = seeded(new Halting());
  const cache: any = new B.Store(lower);
  cache.transaction(crm);
  lower.armed = true;
  const given = cache.transaction(person("cy", "Cy", 7n)); // the upsert comes back partial; the cache layer writes it
  assert(equal(lower.partial.map((frame: any) => frame.remainder.subterm), ["$.mutations[0].upsert"]));
  assert(given.length === 1 && given[0].remainder === null && given[0].errors.length === 0);
  assert(equal(people(cache), [["Cy", 7n, []]]) && people(lower).some((each: any) => equal(each, ["Cy", 7n, []])));
}

<a id="BLW-07"></a>
## BLW-07 · Others' writes in: what another commits below, the layer below's follow gives the cache layer as a one-transaction journal, which it applies to what it holds alone, in the order the layer below committed: an object made under one it holds is made in it, and one made under one it does not hold is not, nor changed; a property of one it holds is set; what it does not hold changes nothing; an entry linking one it does not hold is left out, and the objects it holds that the entry links it holds not all of from then; its followers are told what changed, under the id it arrived with

In [ ]:
{
  const [lower, cache] = stacked();
  const journals: any[] = [];
  Changes.observe(cache, (journal: any) => journals.push(journal));
  lower.transaction(person("cy", "Cy")); // made by another below: made above, under the book it holds
  lower.transaction(aged("Cy", 5n)); // a property of one it holds: set
  lower.transaction(aged("Ada", 50n)); // one it does not hold: nothing
  lower.transaction((t: any) => t.uid("rex/1").mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols(
    { x: t.crm.Person }).spec([(_: any) => _.x.name("Ada")])]).then([(w: any) => w.upsert((u: any) => u.exists([
    (p: any) => p.symbols({ x: t.crm.Person, r: t.crm.Pet }).spec([
      (_: any) => _.x.pets({ rex: (e: any) => e.pet(_.r) })])]))]))])); // made under one it does not hold: nothing
  lower.transaction((t: any) => t.uid("rex/2").mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols(
    { r: t.crm.Pet })]).then([(w: any) => w.update((u: any) => u.apply([(p: any) => p.symbols({ r: t.crm.Pet }).spec([
    (_: any) => _.r.kind("dog")])]))]))])); // one under one it does not hold, changed: nothing
  assert(equal(people(cache), [["Cy", 5n, []]]) && journals.length === 2);
  assert(equal(journals.flatMap((journal) => journaled(cache, journal).map((t: any) => t.id)), ["person/cy", "aged/Cy"]));
  cache.transaction(() => assert(cache._knows(cache.singleton("crm.TheBook").contacts({ key: "cy" }))));
  lower.transaction(knows("Cy", "Ada")); // an entry linking Ada, not held above: left out, and Cy held not all of
  cache.transaction(() => {
    const cy = cache.singleton("crm.TheBook").contacts({ key: "cy" });
    assert([...cy.knows].length === 0 && !cache._knows(cy));
  });
  assert(journals.length === 2 && people(lower).some((each: any) => equal(each, ["Cy", 5n, ["Ada"]])));
}

<a id="BLW-08"></a>
## BLW-08 · Layers stack: a cache layer over a cache layer over a store; a transaction goes down to the lowest layer and its frames come back up, each layer applying their journals to what it holds; what another writes to the lowest reaches each, through each follow

In [ ]:
{
  const [lower, middle] = stacked();
  const top: any = new B.Store(middle);
  top.transaction(crm);
  const given = top.transaction(person("cy", "Cy", 3n));
  assert(equal(people(top), [["Cy", 3n, []]]) && equal(people(middle), [["Cy", 3n, []]])
    && people(lower).some((each: any) => equal(each, ["Cy", 3n, []])));
  assert(given[0].journal !== null && given[0].remainder === null);
  const twin = seeded(new B.Store());
  twin.transaction(person("cy", "Cy", 3n));
  assert(paths(top, top.transaction(everyone)) === paths(twin, twin.transaction(everyone)));
  lower.transaction(aged("Cy", 4n)); // another's write, to the lowest
  assert(equal(people(top), [["Cy", 4n, []]]) && equal(people(middle), [["Cy", 4n, []]]));
}

<a id="BLW-09"></a>
## BLW-09 · The completeness rule: a cache layer interpreting read-only binds and reads only what it can know it holds all of; elsewhere the frame halts, and the layer below binds it. A symbol of a schema halts, the cache layer holding no whole extent; a singleton binds, but its spec halts, the cache layer holding not all of it; a reference binds where the cache layer holds its object, and its spec is read there where it holds all of it, halting where it holds not all of it, or none; an update or an upsert halts. Whatever halts, a cache layer gives the frames a store with no layer below holding the same gives, never fewer

In [ ]:
{
  const [lower, cache] = stacked();
  const twin = seeded(new B.Store());
  for (const store of [cache, twin]) store.transaction(person("cy", "Cy", 9n)); // made through the cache layer: held all of
  lower.transaction(person("dan", "Dan")); // made below by another, under the book it holds: held all of too
  twin.transaction(person("dan", "Dan"));
  lower.transaction(knows("Dan", "Ada")); // Dan knows Ada, not held: Dan held not all of from then
  twin.transaction(knows("Dan", "Ada"));
  /** A map binding x to the contact `key` by its reference, whose spec gives its name. */
  const named = (store: any, key: string, name: string) => (t: any): any => t.uid(`named/${key}`).mutations([
    (mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols({ x: contact(store, key) }).spec([
      (_: any) => _.x.name(name)])]))]);
  /** `named`, then an update writing the name it gave. */
  const renamed = (store: any, key: string, name: string) => (t: any): any => t.uid(`renamed/${key}`).mutations([
    (mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols({ x: contact(store, key) }).spec([
      (_: any) => _.x.name(name)])]).then([(w: any) => w.update((u: any) => u.apply([(p: any) => p.symbols(
      { x: contact(store, key) }).spec([(_: any) => _.x.name(name)])]))]))]);
  /** A map binding b to the book, then one whose spec reads its contacts. */
  const book = (t: any): any => t.uid("book/1").mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols(
    { b: t.crm.Book })]).then([(w: any) => w.map((m: any) => m.symbols([(p: any) => p.symbols(
    { b: t.crm.Book, c: t.crm.Person }).spec([(_: any) => _.b.contacts([_.c])])]))]))]);
  const cases: [string, (store: any) => unknown, string | null][] = [ // where read-only the cache layer halts it
    ["a schema's objects", () => everyone, "$.mutations[0].map"],
    ["a singleton, its spec", () => book, "$.mutations[0].map.then[0].map"],
    ["a reference held all of", (store) => named(store, "cy", "Cy"), null],
    ["a reference held all of, its spec not holding", (store) => named(store, "cy", "Zoe"), null],
    ["a reference held not all of", (store) => named(store, "dan", "Dan"), "$.mutations[0].map"],
    ["a reference not held", (store) => named(store, "ada", "Ada"), "$.mutations[0].map"],
    ["an update, of what it holds all of", (store) => renamed(store, "cy", "Cy"), "$.mutations[0].map.then[0].update"],
    ["an upsert", () => person("cy", "Cy", 9n), "$.mutations[0].upsert"],
  ];
  for (const [label, expression, halts] of cases) {
    const local = cache.transaction((txn: any = null) => frames(items(txn.exec(new Frame(), expression(cache), false))));
    assert(equal(local.map((frame: any) => frame.remainder?.subterm ?? null), label.endsWith("not holding") ? [] : [halts]), label);
    const [written, alone] = [cache.transaction(expression(cache)), twin.transaction(expression(twin))];
    assert(paths(cache, written) === paths(twin, alone), label); // the same frames: never fewer
  }
  assert(equal(people(lower), people(twin)));
  const given = new Map([["x", contact(cache, "ada")]]); // an input frame binding a reference the cache layer does not hold
  const dropping = () => (t: any): any => t.uid("dropping/1").mutations([(mu: any) => mu.reduce((r: any) => r.symbols([
    (c: any) => c.symbols(["x"])]))]);
  for (const [expression, halts] of [[(store: any) => named(store, "ada", "Ada"), "$.mutations[0].map"],
    [dropping, "$.mutations[0].reduce"]] as [(store: any) => unknown, string][]) {
    const [local] = cache.transaction((txn: any = null) => frames(items(txn.exec(given, expression(cache), false))));
    assert(local.remainder.subterm === halts && paths(cache, [local]) === paths(cache, [given])); // kept, unresolved
    const written = cache.transaction((txn: any = null) => frames(items(txn.exec(given, expression(cache), true))));
    const alone = twin.transaction((txn: any = null) => frames(items(txn.exec(new Map([["x", contact(twin, "ada")]]),
      expression(twin), true))));
    assert(paths(cache, written) === paths(twin, alone));
  }
}

<a id="BLW-10"></a>
## BLW-10 · A cache layer's own writes, by builders in its transaction, go through as they are made, the layer below's transaction executing each, so that the layer below holds them within the transaction; an object made without its owner is refused; a transaction is recorded once, in the lowest layer, which the cache layer sends its uid; what the layer below reports of it as it commits is taken into the cache layer's, whose followers are given one journal of it, the record among it

In [ ]:
{
  const [lower, cache] = stacked();
  cache.transaction(Provenance.install); // installs run below first, then in the cache layer
  cache.transaction((t: any) => Provenance.extend(t, [t.crm.Person]));
  assert(lower.registered("Provenance.CreatedBy").name === "Provenance.CreatedBy");
  const journals: any[] = [];
  Changes.observe(cache, (journal: any) => journals.push(journal));
  const uid = cache.transaction(() => {
    cache.crm.Person((p: any) => p.book((a: any) => a.insert((e: any) => e.key("gus").book(cache.singleton("crm.TheBook"))))
      .name("Gus"));
    assert(lower.singleton("crm.TheBook").contacts({ key: "gus" }).name.get() === "Gus"); // below, within it
    return Provenance._uid(cache, cache._proposals);
  });
  raises(ValueError, () => cache.transaction(() => {
    cache.crm.Person((p: any) => p.name("Nobody")); // no owner: it cannot go through as it is made
  }), "give it one in the same write");
  const recorded = lower.transaction(() => [...lower.singleton("Provenance.Transactions").transactions]
    .map((e: any) => e.uid.get()));
  const held = cache.transaction(() => [...cache.singleton("Provenance.Transactions").transactions]
    .map((e: any) => e.uid.get()));
  assert(journals.length === 1);
  const [transaction] = journaled(cache, journals[0]);
  assert(equal(recorded, [uid]) && equal(held, [uid]) && transaction.id === uid); // recorded once, below
  const adjacencies = new Set(transaction.changes.filter((change: any) => "entry" in change)
    .map((change: any) => change.entry.insert.adjacency));
  assert(["contacts", "created", "transactions"].every((name) => adjacencies.has(name))); // the record the lowest made
  assert(equal(people(cache), [["Gus", null, []]]) && people(lower).some((each: any) => equal(each, ["Gus", null, []])));
}

<a id="BLW-11"></a>
## BLW-11 · Split-join: a frame that halts within a branch of a partition goes below with the rest of that branch alone; the cache layer, which split it, collects what each branch ends with, joins it, and goes on with the terms after the join; the frames are those a store with no layer below gives; a branch's rest the layer below fails is annotated, the scope then raising it

In [ ]:
{
  const [lower, cache] = stacked();
  const twin = seeded(new B.Store());
  for (const store of [cache, twin]) store.transaction(person("cy", "Cy", 9n)); // held all of by the cache layer
  /** Cy, by her reference, then a partition: its first branch ages her, its second keeps her as she is; then the
   * book. */
  const split = (store: any) => {
    const arm = (then: unknown) => (e: any) => e.branch((b: any) => b.then([then]));
    const cy = contact(store, "cy");
    return (t: any): any => t.uid("split/1").mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols(
      { x: cy })]).then([(w: any) => w.partition((q: any) => q.branches([
      arm((w2: any) => w2.update((u: any) => u.apply([(p: any) => p.symbols({ x: cy }).spec([(_: any) => _.x.age(10n)])]))),
      arm((w2: any) => w2.map((m2: any) => m2.symbols([(p: any) => p.symbols({ x: cy }).spec([(_: any) => _.x.name("Cy")])]))),
    ]).then([(w3: any) => w3.map((m3: any) => m3.symbols([(p: any) => p.symbols({ bk: t.crm.Book })]))]))]))]);
  };
  const sent: string[] = [];
  const original = Transactions._Lower.prototype.forward;
  Transactions._Lower.prototype.forward = function (frame: any, rest: any): any {
    sent.push(rest.subterm);
    return original.call(this, frame, rest);
  };
  let given: any[];
  try {
    given = cache.transaction(split(cache));
  } finally {
    Transactions._Lower.prototype.forward = original;
  }
  assert(equal(sent, ["$.mutations[0].map.then[0].partition.branches[0].then[0].update", "$", "$"])); // a branch's rest;
  // then each frame joined and gone on with, complete, with nothing left to execute
  assert(paths(cache, given) === paths(twin, twin.transaction(split(twin))) && given.length === 2);
  assert(given.every((frame: any) => frame.has("bk")) && equal(people(lower), people(twin))
    && equal(people(cache), [["Cy", 10n, []]]));
  const CY = { singleton: "crm.TheBook", next: { entry: { i: "contacts", o: "book", properties: { key: "cy" },
    next: { target: {} } } } };
  const NOBODY = { singleton: "crm.TheBook", next: { entry: { i: "contacts", o: "book", properties: { key: "zed" },
    next: { target: {} } } } };
  const failing = { uid: "split/3", mutations: [{ map: { symbols: [{ symbols: { x: { reference: { path: CY } } } }], then: [
    { partition: { branches: [{ then: [{ update: { apply: [{ symbols: { x: { reference: { path: CY } } }, spec: [
      { x: [{ as_adjacency: { name: "knows", mutations: [{ insert: { entries: [{ mutations: [{ as_link: {
        name: "known", mutations: [{ set: { reference: { path: NOBODY } } }] } }] }] } }] } }] }] }] } }] }] } }] } }] };
  const seen: any[] = [];
  raises(LookupError, () => cache.transaction((txn: any = null) => { // a branch's rest the layer below fails: annotated
    seen.push(...frames(items(txn.exec(new Frame(), Plain.FromPlain(cache).Transaction(Json.loads(JSON.stringify(failing))),
      true))));
  }), "no object of the store has the path a spec refers to");
  assert(equal(seen.flatMap((frame: any) => frame.errors.map((error: any) => error.subterm)),
    ["$.mutations[0].map.then[0].partition.branches[0].then[0].update"]) && equal(people(cache), [["Cy", 10n, []]]));
}

<a id="BLW-12"></a>
## BLW-12 · What cannot pass between layers is refused, not lost: a counted object, which has no reference, binds a frame the layer below gives back refused, as any frame given back binds one; given to the cache layer to send below, the frame is annotated; a schema object the cache layer does not hold, bound below, binds the frame it comes back in refused

In [ ]:
{
  const [lower, cache] = stacked();
  const note = (t: any): any => t.uid("note/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols(
    { n: t.crm.Note }).spec([(_: any) => _.n.text("hi")])]))]);
  const retext = (t: any): any => t.uid("retext/1").mutations([(mu: any) => mu.update((u: any) => u.apply([(p: any) => p.symbols(
    { n: t.crm.Note }).spec([(_: any) => _.n.text("ho")])]))]);
  let sent: any = null;
  raises(LookupError, () => cache.transaction((txn: any = null) => {
    const [noted] = frames(items(txn.exec(new Frame(), note, true))); // made below: it comes back refused
    raises(LookupError, () => noted.get("n"), "symbol 'n' of the frame: a counted 'crm.Note' has no reference");
    [sent] = frames(items(txn.exec(noted, retext, true))); // refused here, as any frame given back binds one
  }), "has no reference");
  assert(sent.errors[0].message.startsWith("symbol 'n' of the frame: a counted 'crm.Note' has no reference"));
  lower.transaction((t: any) => t.uid("lab/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols(
    { Rack: t.Schemas.OfObject }).spec([(_: any) => _.Rack.name("lab.Rack").ref(true)])]))])); // below alone
  const racks = (t: any): any => t.uid("racks/1").mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols(
    { s: t.Schemas.OfObject }).spec([(_: any) => _.s.name("lab.Rack")])]))]);
  const [rack] = cache.transaction(racks);
  raises(LookupError, () => rack.get("s"), "symbol 's' of the frame: a schema object the layer it goes to does not hold");
}

<a id="BLW-13"></a>
## BLW-13 · The cache layer settles and checks its own transaction first: a refusal there, a unique clause its own write breaks, refuses its transaction before the layer below's commits, which is dropped, never committing; then, of a transaction that commits, the layer below commits, then the cache layer

In [ ]:
{
  const [lower, cache] = stacked();
  cache.transaction(person("cy", "Cy")); // held all of by the cache layer
  const committing: boolean[] = [];
  const spy = (proposals: unknown): void => {
    committing.push(cache._proposals !== null);
    B.Store.prototype._settled.call(lower, proposals as any);
  };
  lower._settled = spy;
  raises(ValueError, () => cache.transaction(() => { // a second contact keyed "cy", the cache layer's own check refuses
    cache.crm.Book(cache.singleton("crm.TheBook"), (b: any) => b.contacts((a: any) => a.insert(
      (e: any) => e.key("cy").contact((c: any) => c.name("Cee")))));
  }), "breaks unique");
  assert(equal(committing, []) && lower._proposals === null); // the layer below's transaction dropped, never committing
  delete lower._settled;
  assert(equal(people(lower).map((each: any) => each[0]), ["Ada", "Bob", "Cy"]) && equal(people(cache), [["Cy", null, []]]));
  lower._settled = spy;
  cache.transaction(aged("Cy", 4n)); // then, a transaction that commits: the layer below's first, the cache layer's open
  delete lower._settled;
  assert(equal(committing, [true]) && people(lower).some((each: any) => equal(each, ["Cy", 4n, []]))
    && equal(people(cache), [["Cy", 4n, []]]));
}

<a id="BLW-14"></a>
## BLW-14 · Others' writes in, each object named by its reference: a journal the layer below gives, whose entry changes name their holders and links by object reference, is applied to the objects the cache layer holds alone: an entry whose every object it holds is written, one naming an object it does not hold is left out, and the objects it holds that such an entry names it holds not all of from then; one that names none it holds changes nothing; a parameter reference in an object's place is kept


In [ ]:
{
  const [, cache] = stacked();
  cache.transaction(person("cy", "Cy")); // each held all of by the cache layer
  cache.transaction(person("dee", "Dee"));
  const journals: any[] = [];
  Changes.observe(cache, (journal: any) => journals.push(journal));
  const at = (key: string): any => ({ path: { singleton: "crm.TheBook", next: { entry: {
    i: "contacts", o: "book", properties: { key }, next: { target: {} } } } } });
  cache._arrive(map([{ id: "refs/1", changes: [
    { entry: { insert: { holder: { reference: at("cy") }, adjacency: "knows",
      links: { known: { reference: at("dee") } } } } }, // both held: written
    { entry: { insert: { holder: { reference: at("dee") }, adjacency: "knows",
      links: { known: { reference: at("ada") } } } } }] }])); // Ada not held: left out
  assert(equal(people(cache), [["Cy", null, ["Dee"]], ["Dee", null, []]]));
  cache.transaction(() => {
    const book = cache.singleton("crm.TheBook");
    assert(cache._knows(book.contacts({ key: "cy" })) && !cache._knows(book.contacts({ key: "dee" })));
  });
  const [transaction] = journaled(cache, journals.at(-1));
  assert(transaction.id === "refs/1" && transaction.changes.length === 1); // under the id it arrived with
  cache._arrive(map([{ id: "refs/2", changes: [{ entry: { insert: { holder: { reference: at("ada") },
    adjacency: "knows", links: { known: { reference: at("bob") } } } } }] }])); // neither held: nothing
  assert(journals.length === 1 && equal(people(cache), [["Cy", null, ["Dee"]], ["Dee", null, []]]));
  cache._arrive(map([{ id: "refs/3", changes: [{ entry: { insert: { holder: { reference: at("cy") },
    adjacency: "knows", links: { known: { parameter: { name: "p" } } } } } }] }])); // a parameter reference: kept
  cache.transaction(() => {
    const names: unknown[] = [];
    for (const entry of cache.singleton("crm.TheBook").contacts({ key: "cy" }).knows) {
      if (entry.known.as_schema() === null) entry.known.as_parameter((p: any) => names.push(p.name.get()));
    }
    assert(equal(names, ["p"]) && journals.length === 2);
  });
}
